# 23 — Method Chaining & Pipes

Method chaining turns imperative, multi-step transformations into declarative,
readable pipelines. This notebook covers the chaining pattern, assign(), pipe(),
and production-quality pipeline design.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

# Raw, messy customer orders dataset
raw_orders = pd.DataFrame({
    'Order_ID':   range(1001, 1001 + n),
    'Customer':   [f'  Customer_{i:03d}  ' for i in np.random.randint(1, 100, n)],
    'Category':   np.random.choice(['ELECTRONICS', 'Clothing', 'food', 'BOOKS', None], n,
                                    p=[0.3, 0.25, 0.2, 0.2, 0.05]),
    'Amount_Raw': [f'${np.random.randint(500, 80000):,}' for _ in range(n)],
    'Status':     np.random.choice(['delivered', 'RETURNED', 'Pending', 'CANCELLED', None], n,
                                    p=[0.5, 0.15, 0.2, 0.1, 0.05]),
    'Date_Str':   pd.date_range('2022-01-01', periods=n, freq='D').strftime('%d/%m/%Y'),
    'Units':      np.random.randint(1, 20, n),
})

print(f"Raw shape: {raw_orders.shape}")
raw_orders.head(3)

## 1. Imperative Style vs Method Chaining

In [ ]:
# ❌ IMPERATIVE: lots of intermediate variables, harder to read
df = raw_orders.copy()
df.columns = df.columns.str.lower().str.replace(' ', '_')
df['category'] = df['category'].str.title().fillna('Unknown')
df['status'] = df['status'].str.title().fillna('Unknown')
df['amount'] = pd.to_numeric(df['amount_raw'].str.replace(r'[\$,]', '', regex=True))
df['order_date'] = pd.to_datetime(df['date_str'], format='%d/%m/%Y')
df = df.drop(columns=['amount_raw', 'date_str'])
df = df[df['amount'] > 0]
print(f"Imperative result: {df.shape}")

In [ ]:
# ✅ CHAINED: single expression, no intermediate state
cleaned = (
    raw_orders
    .rename(columns=lambda c: c.strip().lower().replace(' ', '_'))           # normalize cols
    .assign(
        category   = lambda df: df['category'].str.title().fillna('Unknown'),
        status     = lambda df: df['status'].str.title().fillna('Unknown'),
        amount     = lambda df: pd.to_numeric(
                         df['amount_raw'].str.replace(r'[\$,]', '', regex=True),
                         errors='coerce'
                     ),
        order_date = lambda df: pd.to_datetime(df['date_str'], format='%d/%m/%Y'),
        customer   = lambda df: df['customer'].str.strip(),
    )
    .drop(columns=['amount_raw', 'date_str'])
    .query('amount > 0')
    .reset_index(drop=True)
)

print(f"Chained result: {cleaned.shape}")
print(cleaned.head(3))

## 2. assign() — The Core of Chaining

In [ ]:
# assign() adds or overwrites columns without mutating the original
# Each lambda receives the df AFTER previous assigns in the same call (Pandas 0.23+)

result = (
    cleaned
    .assign(
        revenue      = lambda df: df['amount'] * df['units'],      # new column
        discount     = lambda df: np.where(df['units'] > 10, 0.10, 0),  # conditional
        net_revenue  = lambda df: df['revenue'] * (1 - df['discount']),  # uses 'revenue' from above
        month        = lambda df: df['order_date'].dt.month,
        is_weekend   = lambda df: df['order_date'].dt.day_of_week >= 5,
        tier         = lambda df: pd.cut(
                           df['amount'],
                           bins=[0, 5000, 20000, 80001],
                           labels=['Low', 'Mid', 'High']
                       )
    )
)

print(result[['order_id', 'amount', 'units', 'revenue', 'net_revenue', 'tier']].head(6))

## 3. pipe() — External Functions in a Chain

In [ ]:
# pipe(func) is equivalent to func(df)
# It lets you insert any function into a chain

def add_revenue(df):
    return df.assign(revenue=df['amount'] * df['units'])

def filter_active(df, min_amount=1000):
    return df[df['amount'] >= min_amount]

def log_shape(df, label=''):
    print(f"[{label}] shape: {df.shape}")
    return df  # always return df to keep chain flowing


result = (
    cleaned
    .pipe(log_shape, label='raw')
    .pipe(filter_active, min_amount=5000)
    .pipe(log_shape, label='after filter')
    .pipe(add_revenue)
    .pipe(log_shape, label='with revenue')
)

print(result[['order_id', 'amount', 'units', 'revenue']].head(5))

In [ ]:
# pipe() is especially useful for:
# 1. Debug checkpoints (log_shape above)
# 2. Complex functions that don't fit in assign()
# 3. Reusable pipeline stages

def compute_running_revenue(df):
    """Adds cumulative revenue sorted by date."""
    return df.sort_values('order_date').assign(
        cumulative_revenue=lambda d: d['amount'].cumsum()
    )

with_cumrev = cleaned.pipe(compute_running_revenue)
print(with_cumrev[['order_date', 'amount', 'cumulative_revenue']].tail(5))

## 4. query() in Chains

In [ ]:
# query() is chain-friendly — returns df, accepts @vars
min_rev = 10000
valid_statuses = ['Delivered', 'Pending']

active_high_value = (
    cleaned
    .assign(revenue=lambda df: df['amount'] * df['units'])
    .query("status in @valid_statuses and amount > @min_rev")
    .query("category != 'Unknown'")
    .reset_index(drop=True)
)

print(f"Active high-value orders: {len(active_high_value):,}")
print(active_high_value.head(4))

## 5. sort_values() in Chains

In [ ]:
# Sort and head in a chain
top10 = (
    cleaned
    .assign(revenue=lambda df: df['amount'] * df['units'])
    .query("status == 'Delivered'")
    .sort_values('revenue', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print("Top 10 delivered orders by revenue:")
print(top10[['order_id', 'customer', 'category', 'amount', 'units', 'revenue']])

## 6. groupby() in Chains

In [ ]:
# groupby integrates cleanly in chains
category_summary = (
    cleaned
    .assign(revenue=lambda df: df['amount'] * df['units'])
    .query("status != 'Unknown'")
    .groupby('category')
    .agg(
        orders=('order_id', 'count'),
        total_revenue=('revenue', 'sum'),
        avg_order=('amount', 'mean')
    )
    .round(0)
    .sort_values('total_revenue', ascending=False)
)
print(category_summary)

## 7. Production Pipeline Pattern

In [ ]:
# Production pipeline: each stage is a named function

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns=lambda c: c.strip().lower().replace(' ', '_'))


def parse_fields(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(
        category   = lambda df: df['category'].str.title().fillna('Unknown'),
        status     = lambda df: df['status'].str.title().fillna('Unknown'),
        customer   = lambda df: df['customer'].str.strip(),
        amount     = lambda df: pd.to_numeric(
                         df['amount_raw'].str.replace(r'[\$,]', '', regex=True),
                         errors='coerce'
                     ),
        order_date = lambda df: pd.to_datetime(df['date_str'], format='%d/%m/%Y'),
    ).drop(columns=['amount_raw', 'date_str'])


def drop_invalid(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(subset=['amount']).query('amount > 0')


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(
        revenue    = lambda df: df['amount'] * df['units'],
        year       = lambda df: df['order_date'].dt.year,
        month      = lambda df: df['order_date'].dt.month,
        is_weekend = lambda df: df['order_date'].dt.day_of_week >= 5,
        tier       = lambda df: pd.cut(
                         df['amount'],
                         bins=[0, 5000, 20000, 80001],
                         labels=['Low', 'Mid', 'High'],
                         right=True
                     )
    )


# Full pipeline — clean, testable, readable
processed = (
    raw_orders
    .pipe(normalize_columns)
    .pipe(parse_fields)
    .pipe(drop_invalid)
    .pipe(add_features)
    .reset_index(drop=True)
)

print(f"Processed: {processed.shape}")
print(processed.dtypes)
print(processed.head(3))

## 8. Chain Rules — What to Avoid

In [ ]:
# RULE 1: Never use inplace=True in a chain — breaks the pipeline
# BAD:
# df.sort_values('amount', inplace=True).head(5)  # AttributeError — inplace returns None

# RULE 2: Don't modify external state inside assign()
# BAD: side effects inside lambdas

# RULE 3: Avoid deeply nested lambdas — extract to named functions
# BAD:
# .assign(x=lambda df: df.groupby('cat').transform(lambda g: g.fillna(g.median())))
# GOOD: define compute_x() separately and use .pipe()

print("Chain rules:")
print("  ✅ assign() for new/modified columns")
print("  ✅ pipe() for complex logic")
print("  ✅ query() for filtering")
print("  ❌ inplace=True (returns None, breaks chain)")
print("  ❌ Mutable closures inside lambdas")

## 9. DataFrame.style — Highlighting in Chains

In [ ]:
# .style allows visual highlighting inline (Jupyter only)
summary = (
    processed
    .groupby('category')
    .agg(orders=('order_id', 'count'), total=('revenue', 'sum'), avg=('amount', 'mean'))
    .round(0)
)

# In Jupyter, this renders a color-coded table
# summary.style.background_gradient(cmap='Blues', subset=['total'])
print(summary)